# 04 — Selection verdict

Applies the pre-registered decision rule (`README.md`) to the notebook 01–03 results
on the even half, freezes ONE configuration per channel, and only then evaluates that
single configuration on the **odd half** (the confirmation — these numbers were not
inspected before the choice).

In [1]:
import json, os, sys
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

sys.path.insert(0, os.getcwd())
import abcd_tools as at
import study_setup as ss

hep.style.use("CMS")
plt.rcParams["figure.figsize"] = (7, 5)
LUMI_LABEL = r"59.8 fb$^{-1}$ (13 TeV, 2018 sim.)"

def cms_label(ax=None):
    hep.cms.label("Work in progress", data=False, rlabel=LUMI_LABEL, ax=ax)

# pre-skim / pre-filter sums of gen weights (see README "Normalization")
SUMW_PRE = {}
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_2MU2E))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_SIGNAL_4MU))
SUMW_PRE.update(at.census_sumw_pre(ss.CENSUS_BKG_UNSKIMMED))

# Background sums built ONE SAMPLE AT A TIME (holding all 44 samples in memory OOMs
# the interactive node); signals are loaded lazily where needed, one at a time.
# TTJets kept at the campaign 471.7 pb; the NNLO alternative is an explicit rescale.
total_bkg, by_process = ss.accumulate_normalized(list(ss.BACKGROUNDS), SUMW_PRE)
def load_sig(s):
    return ss.load_normalized(s, SUMW_PRE)[0]
print(f"accumulated {len(ss.BACKGROUNDS)} backgrounds; scan hists: {len(total_bkg)}")

accumulated 18 backgrounds; scan hists: 8


In [2]:
gates = json.load(open(os.path.join(ss.WORKDIR, "gates_even.json")))
sens = json.load(open(os.path.join(ss.WORKDIR, "sensitivity_even.json")))
import collections

# AMENDMENTS to the pre-registered rule, forced by MC statistics and recorded openly:
# (1) gate 1 is evaluated at PRESEL level (tight-WP per-process fits cannot run);
#     a nan p-value for a process with negligible presel yield counts as vacuous-pass.
# (2) gate 2 (statistical health) is re-scoped from the tight WP itself (n_eff ~ 1
#     everywhere -> no plane could ever pass with weighted MC) to the projection
#     trajectory: >= 3 healthy (n_eff > 10) scan points. Data (unweighted) will
#     re-test the tight WPs directly in round 2.
verdict = {}
for ch in ss.PLANES:
    rows = []
    for pname in list(ss.PLANES[ch]) + list(ss.DERIVED_PLANES.get(ch, {})):
        key = f"{ch}/{pname}"
        pass1 = False
        for presc1 in ("i", "ii"):
            g1p = gates["gate1"].get(f"{key}/{presc1}", {})
            finite = [v for v in g1p.values() if v is not None and np.isfinite(v)]
            if finite and all(v > 0.05 for v in finite):
                pass1 = True
        screened = pname in ss.SCREENED_ONLY[ch] and pname in ss.PLANES[ch]
        best = None
        for presc in ["iii", "i"]:   # prefer the sentinel-excluded SR definition
            g3 = gates["gate3"].get(f"{key}/{presc}") or {}
            if "anchor_R" not in g3:
                continue
            healthy = g3.get("n_healthy", 0) >= 3
            closed = bool(g3.get("anchor_pass"))
            if best is None or (healthy and closed and not best[1]):
                best = (presc, healthy and closed, g3)
        pass2 = best is not None and best[2].get("n_healthy", 0) >= 3
        pass3 = None if screened else (best is not None and best[1])
        zs = [v["Z"] for k, v in sens.items()
              if k.startswith(f"{key}/{best[0]}/") ] if best else []
        g2rec = gates["gate2"].get(key)
        tight_neff = min(g2rec["n_eff"].values()) if g2rec else np.nan
        rows.append((pname, pass1, pass2, pass3, screened,
                     float(np.median(zs)) if zs else np.nan, tight_neff,
                     best[0] if best else "-",
                     best[2].get("anchor_R") if best else np.nan,
                     best[2].get("anchor_err") if best else np.nan))
    print(f"=== {ch}")
    print(f"{'plane':18s} {'G1':>6s} {'G2':>6s} {'G3':>7s} {'presc':>6s} "
          f"{'R_anchor':>14s} {'medZ':>7s} {'tightWP_neff':>13s}")
    for r in rows:
        g3s = "screen" if r[4] else str(r[3])
        rp = f"{r[8]:.3f}+-{r[9]:.3f}" if np.isfinite(r[8] or np.nan) else "-"
        print(f"{r[0]:18s} {str(r[1]):>6s} {str(r[2]):>6s} {g3s:>7s} {r[7]:>6s} "
              f"{rp:>14s} {r[5]:7.3f} {r[6]:13.1f}")
    surv = [r for r in rows if r[1] and r[2] and r[3]]
    if surv:
        best_row = max(surv, key=lambda r: (r[5] if np.isfinite(r[5]) else -1, -abs((r[8] or 2) - 1)))
        verdict[ch] = (best_row[0], best_row[7])
        print(f"--> chosen: {best_row[0]} with prescription ({best_row[7]})")
    else:
        verdict[ch] = None
        print("--> NO SURVIVOR under the amended gates")

=== 2mu2e
plane                  G1     G2      G3  presc       R_anchor    medZ  tightWP_neff
P1_iso_iso          False   True   False    iii   0.454+-0.196   0.033           1.1
P2_muiso_dphi        True  False   False    iii   2.118+-0.815   0.001           1.1
P3_egmiso_dphi      False  False   False      -              -     nan           1.1
P4_muiso_mjj         True  False   False    iii   0.826+-0.283   0.166           1.0
P5_muiso_mupix       True  False  screen      -              -     nan           1.1
P6_mupix_dphi       False  False  screen      -              -     nan           1.1
P7_egmlost_dphi     False  False  screen      -              -     nan           1.0
P8_dphi_mjj         False  False   False    iii   3.031+-0.937   0.010           1.0
D1_ntight_dphi       True  False   False      -              -     nan           nan
D2_ntight_mjj        True  False   False      -              -     nan           nan
D3_jetmatch_dphi    False  False   False      -        

### The 4mu channel

The expected 4mu SR background at 59.8 fb$^{-1}$ is at the few-permille level — the
channel is effectively background-free at the incumbent working points. An ABCD ratio
is neither needed nor stable there; the recommendation is a counting treatment (the
looser-boundary projection above validates the background model where statistics
exist, and the SR expectation enters the limit as a Poisson mean with the projection
systematic). This is quantified in the cell below.

In [3]:
for ch, pname in [("4mu", "Q1_iso_iso")]:
    spec = ss.PLANES[ch][pname]
    vals, var, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
    for lab, kw in [("prescription (i)", {}), ("prescription (iii)", dict(xlo=0.0, ylo=0.0))]:
        reg = at.region_sums(vals, var, xe, ye, spec["xspec"], spec["yspec"], **kw)
        pred, pvar = at.abcd_prediction(reg)
        a_obs = reg["A"][0]
        print(f"{ch}/{pname} {lab}: A_obs(MC) = {a_obs:.4g}, "
              f"BC/D = {pred:.4g} +- {np.sqrt(max(pvar,0)):.4g}")

4mu/Q1_iso_iso prescription (i): A_obs(MC) = 0.002632, BC/D = 0.0438 +- 0.05568
4mu/Q1_iso_iso prescription (iii): A_obs(MC) = 0, BC/D = 0.04364 +- 0.05559


## Plateau check and odd-half confirmation

The chosen working point must give the same verdict under ±1-bin boundary shifts;
then the frozen configuration is evaluated once on the odd half.

In [4]:
for ch, chosen in verdict.items():
    if chosen is None:
        continue
    pname, presc = chosen
    if pname not in ss.PLANES[ch]:
        print(f"{ch}: chosen plane {pname} is categorical - continuous plateau check "
              f"N/A; its robustness statement is the event-cut ladder in notebook 02")
        continue
    lo = dict(xlo=0.0, ylo=0.0) if presc == "iii" else {}
    spec = ss.PLANES[ch][pname]
    # plateau: +-1 bin on each boundary
    vals, var, xe, ye = ss.plane_arrays(total_bkg, ch, pname, parity=0)
    ix = at.edge_index(xe, spec["xspec"][1]); iy = at.edge_index(ye, spec["yspec"][1])
    print(f"--- {ch}/{pname} plateau (even):")
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            reg = at.region_sums(vals, var, xe, ye,
                                 (spec["xspec"][0], float(xe[ix + dx])),
                                 (spec["yspec"][0], float(ye[iy + dy])), **lo)
            r, vr = at.closure_ratio(reg)
            print(f"  ({dx:+d},{dy:+d}): R = {r:6.3f} +- {np.sqrt(max(vr,0)):5.3f}")
    # the one look at the odd half
    ovals, ovar, oxe, oye = ss.plane_arrays(total_bkg, ch, pname, parity=1)
    oreg = at.region_sums(ovals, ovar, oxe, oye, spec["xspec"], spec["yspec"], **lo)
    orr, ovr = at.closure_ratio(oreg)
    print(f"  ODD-HALF confirmation: R = {orr:6.3f} +- {np.sqrt(max(ovr,0)):5.3f}, "
          f"A = {oreg['A'][0]:9.4g}")

## Findings and recommendation (2018 MC round)

**What the MC can and cannot establish.** No candidate plane passes the full
pre-registered gate set at the analysis working points: every tight region is
dominated by single large-weight low-pT QCD events (effective counts ≈ 1), so
weighted MC cannot validate tight-WP closure directly for ANY plane. This is a
quantified statement, not a failure: the decisive closure test must be data-driven
(data sidebands are unweighted — thousands of effective events where the MC has one),
which is the round-2 plan. What the MC does establish, with full presel statistics:

1. **The incumbent iso×iso plane (P1) should not be used as-is.** Total-background
   factorization fails at preselection (p = 0.006; DY alone p = 0.007, and the
   QCD/DY/TTJets mixture breaks factorization even where each process passes). Its
   presel closure is R = 0.45 ± 0.20, and the all-sidebands estimator confirms the
   over-prediction with tighter errors. This independently corroborates — and
   sharpens — the choice to re-examine the production plane.
2. **muiso × mJJ (P4) is the leading candidate.** Every process AND the total
   factorize (p ≈ 0.9); presel closure R = 0.83 ± 0.28 at loose boundaries. mJJ as a
   plane axis also gives the low-mass/high-mass two-region search for free, instead
   of discarding mJJ < 150 events.
3. **muiso × |Δφ| (P2) is disfavored**: its presel non-closure (R ≈ 2) is real, not
   statistical — the all-sidebands estimator reproduces it.
4. **The isolation quirk is a structural feature of the selection, not a binning
   detail.** Failed-jet-match ("sentinel") events are 95–100% of the naive tight-SR
   background and are constrained by NO isolation sideband; excluding them
   (equivalently: requiring the leading LJs to have a matched AK4 jet) costs 0–4% of
   short/mid-cτ signal but 30–50% at the longest lifetimes. Recommendation: make the
   jet-matched SR the baseline and treat the no-jet population as an explicitly
   separate category in the data round (it needs a non-isolation discriminant).
5. **4mu is effectively background-free** at these working points (plain ABCD
   0.044 ± 0.056 expected in the even half). Recommendation: counting-experiment
   treatment, with the ladder as the background-model validation at looser cuts.
6. **Estimator upgrade candidate**: the extended-ABCD / per-LJ fake-factor estimator
   (pass rates measured from all sidebands simultaneously) is unbiased with smaller
   variance than B·C/D on toys and reproduces it at preselection. Its current
   implementation is unstable in the ultra-sparse tight MC regions, so it is a
   data-round upgrade — sparsity disappears there — with plain ABCD as cross-check.

**Carried to round 2 (data)**: P4-style plane + jet-matched SR + two mJJ regions;
the ladder re-run on data sidebands to set the closure systematic at the real WPs;
the low-mJJ region as the designated validation region; cosmic veto (data-only
pathology) enters there; TTJets 471.7 → 831.76 pb pending sign-off (composition
effect shown small in notebook 01).